In [5]:
from pathlib import Path
from collections import defaultdict
from PIL import Image
import hashlib
import imagehash

In [6]:
pwd

'/Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Baseline  notebooks'

In [7]:
#directory path
dataset_path = Path("/Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_Images/Segmentation/semantic level masks/images")

splits = ["train_split", "val_split", "test_split"]
image_extensions = {".png"}


In [8]:
#COLLECT ALL IMAGES
images = []

for split in splits:
    split_dir = dataset_path / split

    if not split_dir.exists():
        print(f"WARNING: {split_dir} does not exist")
        continue

    for path in split_dir.rglob("*"):
        if path.is_file() and path.suffix.lower() in image_extensions:
            images.append({
                "path": path,
                "split": split
            })

print(f"Total images found: {len(images)}")

for split in splits:
    count = sum(1 for x in images if x["split"] == split)
    print(f"{split}: {count}")

Total images found: 686
train_split: 549
val_split: 69
test_split: 68


In [9]:
def calculate_image_hash(image_path):
    """Calculate SHA-256 hash of the given image file."""
    with Image.open(image_path) as img:
        # Convert image to RGB and resize to a consistent size for hashing
        img = img.convert("RGB").resize((256, 256))
        # Get image data as bytes
        img_data = img.tobytes()
        # Create a SHA-256 hash object
        hash_obj = hashlib.sha256()
        # Update the hash object with the image data
        hash_obj.update(img_data)
        # Get the hexadecimal hash string
        return hash_obj.hexdigest()

In [10]:
# FUNCTION TO GENERATE PERCEPTUAL HASHES

def generate_hashes(dataset_path):
    image_hashes = []
    for split in splits:
        split_path = dataset_path / split
        if not split_path.exists():
            print(
                f"WARNING: Folder not found: "
                f"{split_path}"
            )
            continue
        # rglob() searches through the split and
        # any class/subfolders inside it.
        for image_path in split_path.rglob("*"):
            if (
                image_path.is_file()
                and image_path.suffix.lower()
                in image_extensions
            ):
                try:
                    image = Image.open(image_path)
                    image_hash = imagehash.phash(image)
                    image_hashes.append({
                        "path": image_path,
                        "split": split,
                        "hash": image_hash
                    })
                    image.close()
                except Exception as e:
                    print(
                        f"Error processing "
                        f"{image_path}: {e}"
                    )
    return image_hashes


In [11]:
# Generate hashes for the set of images
image_hashes = generate_hashes(dataset_path)

In [12]:
image_hashes

[{'path': PosixPath('/Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_Images/Segmentation/semantic level masks/images/train_split/Rhodes-University-Location-1-Image-Data (40)_counter_181.png'),
  'split': 'train_split',
  'hash': array([[ True, False,  True,  True,  True, False, False,  True],
         [ True,  True, False, False, False,  True,  True, False],
         [False, False,  True,  True,  True, False,  True,  True],
         [ True, False, False,  True,  True, False,  True,  True],
         [ True,  True, False, False, False,  True, False, False],
         [False, False,  True, False,  True,  True, False, False],
         [False, False,  True, False, False,  True, False, False],
         [ True,  True, False, False,  True,  True,  True, False]])},
 {'path': PosixPath('/Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_Images/Segmentation/semantic level 

In [13]:
# FUNCTION TO SEARCH FOR NEAR-DUPLICATE IMAGES

def search_similar_images(
    image_hashes,
    threshold=6
):

    similar_images = []
    for i in range(len(image_hashes)):
        for j in range(i + 1, len(image_hashes)):
            image1 = image_hashes[i]
            image2 = image_hashes[j]
            # Hamming distance between perceptual hashes
            distance = (
                image1["hash"] -
                image2["hash"]
            )
            if distance <= threshold:
                similar_images.append({
                    "image1": image1["path"],
                    "split1": image1["split"],
                    "image2": image2["path"],
                    "split2": image2["split"],
                    "distance": distance
                })
    return similar_images

In [14]:
# GENERATE PERCEPTUAL HASHES


print("\n==============================================")
print("PERCEPTUAL HASHING")
print("==============================================")

image_hashes = generate_hashes(
    dataset_path
)

print(
    f"Images successfully hashed: "
    f"{len(image_hashes)}"
)



PERCEPTUAL HASHING
Images successfully hashed: 686


In [15]:

# SEARCH FOR NEAR-DUPLICATES

# Hamming distance threshold.
# Smaller values indicate greater similarity.
#
# These are candidate near-duplicates and should be
# manually inspected before being classified as genuine
# near-duplicates.

PHASH_THRESHOLD = 6

similar_images = search_similar_images(
    image_hashes,
    threshold=PHASH_THRESHOLD
)


In [16]:

# DISPLAY ALL NEAR-DUPLICATE CANDIDATES

print("\n==============================================")
print("NEAR-DUPLICATE ANALYSIS")
print("==============================================")

print(
    f"pHash threshold: "
    f"{PHASH_THRESHOLD}"
)

print(
    f"Near-duplicate candidate pairs: "
    f"{len(similar_images)}"
)


for pair in similar_images:

    print(
        f"\nHamming distance: "
        f"{pair['distance']}"
    )

    print(
        f"  [{pair['split1']}] "
        f"{pair['image1']}"
    )

    print(
        f"  [{pair['split2']}] "
        f"{pair['image2']}"
    )



NEAR-DUPLICATE ANALYSIS
pHash threshold: 6
Near-duplicate candidate pairs: 6

Hamming distance: 6
  [train_split] /Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_Images/Segmentation/semantic level masks/images/train_split/Rhodes-University-Location-1-Image-Data (31)_counter_169.png
  [train_split] /Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_Images/Segmentation/semantic level masks/images/train_split/Rhodes-University-Location-1-Image-Data (32)_counter_168.png

Hamming distance: 4
  [train_split] /Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_Images/Segmentation/semantic level masks/images/train_split/Rhodes-University-Location-1-Image-Data (58)_counter_167.png
  [train_split] /Users/triscillar/Documents/Professional development/Journal Submissions/data MDPI/Wildlife Fences Dataset/Still_I

In [17]:
# IDENTIFY CROSS-SPLIT NEAR-DUPLICATES

cross_split_similar = []
for pair in similar_images:
    if pair["split1"] != pair["split2"]:
        cross_split_similar.append(pair)

In [18]:
# DISPLAY CROSS-SPLIT RESULTS

print("\n==============================================")
print("CROSS-SPLIT NEAR-DUPLICATES")
print("==============================================")

print(
    f"Cross-split near-duplicate pairs: "
    f"{len(cross_split_similar)}"
)
for pair in cross_split_similar:
    print(
        f"\nHamming distance: "
        f"{pair['distance']}"
    )
    print(
        f"  [{pair['split1']}] "
        f"{pair['image1']}"
    )
    print(
        f"  [{pair['split2']}] "
        f"{pair['image2']}"
    )


CROSS-SPLIT NEAR-DUPLICATES
Cross-split near-duplicate pairs: 0


In [19]:

# FINAL SUMMARY

print("\n==============================================")
print("FINAL SUMMARY")
print("==============================================")

print(
    f"Total images analysed:       "
    f"{len(image_hashes)}"
)

print(
    f"Near-duplicate candidates:   "
    f"{len(similar_images)}"
)

print(
    f"Cross-split near-duplicates: "
    f"{len(cross_split_similar)}"
)


FINAL SUMMARY
Total images analysed:       686
Near-duplicate candidates:   6
Cross-split near-duplicates: 0
